In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('02_profiling', 'invoice_lines')

# 02 · Profiling y Control de Calidad

Este notebook realiza validaciones de calidad en los datos RAW extraídos:
- Detección de nulos
- Rangos de valores
- Validación de tipos
- Identificación de anomalías

**Entrada:** Datos en `raw.invoice_lines`  
**Salida:** Reporte en `datos/bronze/` y registros problemáticos en `raw.validacion_calidad`

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_DESTINO
from datetime import datetime
import os
import pandas as pd
import json

engine_origen, engine_destino = obtener_engines()

# Rutas
DATOS_DIR = './datos'
BRONZE_DIR = os.path.join(DATOS_DIR, 'bronze')
os.makedirs(BRONZE_DIR, exist_ok=True)

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Timestamp: {TIMESTAMP}')
print(f'Destino: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Timestamp: 20261103_124046
Destino: WideWorldImportersDW2


## Paso 1: Crear tabla de validación

In [2]:
from sqlalchemy import text

sql_crear_validacion = """
IF OBJECT_ID('raw.validacion_calidad', 'U') IS NULL
BEGIN
    CREATE TABLE raw.validacion_calidad (
        id_validacion       INT PRIMARY KEY IDENTITY(1,1),
        id_raw              INT NOT NULL,
        lote_id             NVARCHAR(50) NOT NULL,
        tipo_error          NVARCHAR(100) NOT NULL,
        descripcion         NVARCHAR(500),
        valor_campo         NVARCHAR(MAX),
        campo_afectado      NVARCHAR(100),
        timestamp_validacion DATETIME DEFAULT GETDATE(),
        es_critico          BIT DEFAULT 0,
        INDEX IX_raw (id_raw),
        INDEX IX_lote (lote_id)
    );
END;
"""

try:
    with engine_destino.begin() as conn:
        conn.execute(text(sql_crear_validacion))
    print('[OK] Tabla raw.validacion_calidad creada')
except Exception as e:
    print(f'[ERROR] {e}')

[OK] Tabla raw.validacion_calidad creada


## Paso 2: Leer datos RAW

In [8]:
try:
    df_raw = pd.read_sql_query(
        """
        SELECT 
            id_raw, lote_id, numero_fila, payload, hash_sha256
        FROM raw.invoice_lines
        ORDER BY id_raw DESC
        """,
        engine_destino
    )
    print(f'[OK] Se leyeron {len(df_raw)} registros RAW')
    print(f'Lotes únicos: {df_raw["lote_id"].nunique()}')
    print(f'\nÚltimos lotes:')
    print(df_raw[['lote_id', 'numero_fila']].drop_duplicates('lote_id'))
except Exception as e:
    print(f'[ERROR] {e}')
    df_raw = None

[OK] Se leyeron 100 registros RAW
Lotes únicos: 1

Últimos lotes:
                lote_id  numero_fila
0  LOTE_20261103_112725          100


## Paso 3: Parsear y validar payloads

In [9]:
if df_raw is not None and len(df_raw) > 0:
    errores_validacion = []
    registros_validos = []
    
    for idx, row in df_raw.iterrows():
        try:
            payload = json.loads(row['payload'])
            id_raw = row['id_raw']
            lote_id = row['lote_id']
            
            # Validación 1: Campos requeridos
            campos_requeridos = ['InvoiceLineID', 'InvoiceID', 'Quantity', 'UnitPrice']
            for campo in campos_requeridos:
                if campo not in payload or payload[campo] is None:
                    errores_validacion.append({
                        'id_raw': id_raw,
                        'lote_id': lote_id,
                        'tipo_error': 'NULO_EN_CAMPO_REQUERIDO',
                        'descripcion': f'Campo requerido {campo} es nulo',
                        'campo_afectado': campo,
                        'valor_campo': str(payload.get(campo)),
                        'es_critico': 1
                    })
                    continue
            
            # Validación 2: Quantity positiva
            try:
                qty = float(payload.get('Quantity', 0))
                if qty < 0:
                    errores_validacion.append({
                        'id_raw': id_raw,
                        'lote_id': lote_id,
                        'tipo_error': 'VALOR_NEGATIVO',
                        'descripcion': 'Cantidad es negativa (posible nota de credito)',
                        'campo_afectado': 'Quantity',
                        'valor_campo': str(qty),
                        'es_critico': 0
                    })
            except:
                pass
            
            # Validación 3: Precio positivo
            try:
                precio = float(payload.get('UnitPrice', 0))
                if precio < 0:
                    errores_validacion.append({
                        'id_raw': id_raw,
                        'lote_id': lote_id,
                        'tipo_error': 'PRECIO_NEGATIVO',
                        'descripcion': 'Precio unitario es negativo',
                        'campo_afectado': 'UnitPrice',
                        'valor_campo': str(precio),
                        'es_critico': 1
                    })
            except:
                pass
            
            registros_validos.append(payload)
        
        except json.JSONDecodeError as e:
            errores_validacion.append({
                'id_raw': row['id_raw'],
                'lote_id': row['lote_id'],
                'tipo_error': 'JSON_INVALIDO',
                'descripcion': f'Payload JSON no válido: {str(e)}',
                'campo_afectado': 'payload',
                'valor_campo': row['payload'][:100],
                'es_critico': 1
            })
    
    print(f'[OK] Validación completada')
    print(f'Registros válidos: {len(registros_validos)}')
    print(f'Errores encontrados: {len(errores_validacion)}')
    
    if errores_validacion:
        print(f'\nDetalle de errores:')
        df_errores = pd.DataFrame(errores_validacion)
        print(df_errores[['tipo_error', 'es_critico']].value_counts())

[OK] Validación completada
Registros válidos: 100
Errores encontrados: 1

Detalle de errores:
tipo_error       es_critico
PRECIO_NEGATIVO  1             1
Name: count, dtype: int64


## Paso 4: Guardar reporte en archivo

In [10]:
if errores_validacion:
    df_errores = pd.DataFrame(errores_validacion)
    
    # Guardar CSV
    archivo_errores = os.path.join(BRONZE_DIR, f'reporte_calidad_{TIMESTAMP}.csv')
    df_errores.to_csv(archivo_errores, index=False, encoding='utf-8')
    print(f'[OK] Reporte guardado: {archivo_errores}')
    
    # Guardar JSON para referencia
    archivo_errores_json = os.path.join(BRONZE_DIR, f'reporte_calidad_{TIMESTAMP}.json')
    with open(archivo_errores_json, 'w', encoding='utf-8') as f:
        json.dump(errores_validacion, f, indent=2, ensure_ascii=False)
    print(f'[OK] Reporte JSON guardado: {archivo_errores_json}')
else:
    print('[OK] Sin errores de validación')

[OK] Reporte guardado: ./datos\bronze\reporte_calidad_20261103_124046.csv
[OK] Reporte JSON guardado: ./datos\bronze\reporte_calidad_20261103_124046.json


## Paso 5: Insertar errores en BD

In [11]:
if errores_validacion:
    df_errores = pd.DataFrame(errores_validacion)
    
    try:
        df_errores.to_sql(
            'validacion_calidad',
            engine_destino,
            schema='raw',
            if_exists='append',
            index=False,
            chunksize=100
        )
        print(f'[OK] {len(df_errores)} errores registrados en raw.validacion_calidad')
    except Exception as e:
        print(f'[ERROR] {e}')
else:
    print('[OK] No hay errores para registrar')

[OK] 1 errores registrados en raw.validacion_calidad


## Paso 6: Estadísticas y Resumen

In [12]:
try:
    sql_resumen = """
    SELECT 
        COUNT(*) as total_errores,
        SUM(CASE WHEN es_critico = 1 THEN 1 ELSE 0 END) as errores_criticos,
        COUNT(DISTINCT lote_id) as lotes_afectados,
        COUNT(DISTINCT tipo_error) as tipos_error
    FROM raw.validacion_calidad
    """
    
    with engine_destino.connect() as conn:
        result = conn.execute(text(sql_resumen)).mappings().first()
    
    print('[OK] Resumen de validación en BD:')
    if result:
        for key, val in result.items():
            print(f'  {key}: {val}')
except Exception as e:
    print(f'[ERROR] {e}')

[OK] Resumen de validación en BD:
  total_errores: 1
  errores_criticos: 1
  lotes_afectados: 1
  tipos_error: 1


In [ ]:
auditoria_etapa.finalizar(globals())